In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [3]:
from preprocessing.prepare_data import process_data
import os

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler


c:\Users\КАРИНА\PythonProjects\Fraud-Detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
@dataclass
class TrainConfig:
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    val_fraction: float = 0.15
    batch_size: int = 1024
    max_epochs: int = 10
    patience: int = 3
    n_trials: int = 10

In [7]:
class DeepMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: list[int],
        dropout: float,
    ) -> None:
        super().__init__()

        layers: list[nn.Module] = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)
    

class ResidualBlock(nn.Module):
    def __init__(self, dim: int, dropout: float) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(x + self.block(x))
    

class ResidualMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        width: int,
        n_blocks: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.BatchNorm1d(width),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.blocks = nn.Sequential(
            *[ResidualBlock(width, dropout) for _ in range(n_blocks)]
        )
        self.head = nn.Linear(width, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        x = self.blocks(x)
        return self.head(x).squeeze(1)


In [9]:
def prepare_arrays():

    def _resolve_path(p: str | None) -> str | None:
        if not p:
            return None
        return p if os.path.isabs(p) else os.path.join(PROJECT_ROOT, p)

    data = process_data(
        train_transaction_path = _resolve_path("data/train_transaction.csv"),
        train_identity_path = _resolve_path("data/train_identity.csv")
    )

    X_train, X_test, y_train, y_test = data

    X_train = X_train.astype(np.float32)
    X_test = X_test.astype(np.float32)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    y_train = np.asarray(y_train, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler


def time_based_train_val_split(
    X: np.ndarray,
    y: np.ndarray,
    val_fraction: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    split_idx = int((1.0 - val_fraction) * len(X))
    X_tr, X_val = X[:split_idx], X[split_idx:]
    y_tr, y_val = y[:split_idx], y[split_idx:]
    return X_tr, X_val, y_tr, y_val


def make_loader(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> torch.utils.data.DataLoader:
    dataset = torch.utils.data.TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
    )

In [10]:
def build_model(
    architecture: Literal["deep_mlp", "residual_mlp"],
    trial: optuna.trial.Trial,
    input_dim: int,
) -> nn.Module:
    if architecture == "deep_mlp":
        n_layers = trial.suggest_int("n_layers", 2, 5)
        hidden_dims = [
            trial.suggest_int(f"hidden_dim_{i}", 128, 1024, step=128)
            for i in range(n_layers)
        ]
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        return DeepMLP(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
        )

    if architecture == "residual_mlp":
        width = trial.suggest_int("width", 128, 1024, step=128)
        n_blocks = trial.suggest_int("n_blocks", 2, 6)
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        return ResidualMLP(
            input_dim=input_dim,
            width=width,
            n_blocks=n_blocks,
            dropout=dropout,
        )

    raise ValueError(f"Unknown architecture: {architecture}")

In [11]:
def evaluate_auc(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
    device: str,
) -> float:
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_targets.append(yb.numpy())

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probs)
    return roc_auc_score(y_true, y_prob)

In [12]:
def train_one_trial(
    trial: optuna.trial.Trial,
    architecture: Literal["deep_mlp", "residual_mlp"],
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    cfg: TrainConfig,
) -> float:
    input_dim = X_tr.shape[1]

    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024, 2048])
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    model = build_model(architecture, trial, input_dim).to(cfg.device)

    pos_count = max(y_tr.sum(), 1.0)
    neg_count = max(len(y_tr) - y_tr.sum(), 1.0)
    pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32, device=cfg.device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = make_loader(X_tr, y_tr, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(X_val, y_val, batch_size=batch_size, shuffle=False)

    best_val_auc = -np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(cfg.max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        val_auc = evaluate_auc(model, val_loader, cfg.device)

        trial.report(val_auc, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg.patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_val_auc

In [14]:
def run_optuna_for_architecture(
    architecture: Literal["deep_mlp", "residual_mlp"],
    X_train_full: np.ndarray,
    y_train_full: np.ndarray,
    cfg: TrainConfig,
) -> optuna.study.Study:
    X_tr, X_val, y_tr, y_val = time_based_train_val_split(
        X_train_full, y_train_full, cfg.val_fraction
    )

    def objective(trial: optuna.trial.Trial) -> float:
        return train_one_trial(
            trial=trial,
            architecture=architecture,
            X_tr=X_tr,
            y_tr=y_tr,
            X_val=X_val,
            y_val=y_val,
            cfg=cfg,
        )

    study = optuna.create_study(
        direction="maximize",
        study_name=f"{architecture}_study",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )

    study.optimize(objective, n_trials=cfg.n_trials, show_progress_bar=True)
    return study

In [ ]:
def fit_final_model(
    architecture: Literal["deep_mlp", "residual_mlp"],
    best_params: dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    cfg: TrainConfig,
    threshold: float = 0.5,
) -> tuple[np.ndarray, np.ndarray]:
    input_dim = X_train.shape[1]

    # rebuild model from best params
    trial_like = optuna.trial.FixedTrial(best_params)
    model = build_model(architecture, trial_like, input_dim).to(cfg.device)

    batch_size = best_params["batch_size"]
    lr = best_params["lr"]
    weight_decay = best_params["weight_decay"]

    pos_count = max(y_train.sum(), 1.0)
    neg_count = max(len(y_train) - y_train.sum(), 1.0)
    pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32, device=cfg.device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = make_loader(X_train, y_train, batch_size=batch_size, shuffle=True)
    test_loader = make_loader(X_test, np.zeros(len(X_test), dtype=np.float32), batch_size=batch_size, shuffle=False)

    best_loss = np.inf
    best_state = None
    patience_counter = 0

    # small internal holdout from train for final early stopping
    X_tr, X_val, y_tr, y_val = time_based_train_val_split(X_train, y_train, cfg.val_fraction)
    train_loader = make_loader(X_tr, y_tr, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(X_val, y_val, batch_size=batch_size, shuffle=False)

    for _ in range(cfg.max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # use validation loss for final stopping
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                val_loss = criterion(logits, yb).item()
                val_losses.append(val_loss)

        mean_val_loss = float(np.mean(val_losses))

        if mean_val_loss < best_loss:
            best_loss = mean_val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg.patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # predict on test
    model.eval()
    test_probs = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(cfg.device)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            test_probs.append(probs)

    y_proba = np.concatenate(test_probs)
    y_pred = (y_proba >= threshold).astype(np.int32)

    return y_pred, y_proba

In [19]:
def print_metrics(name: str, y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray) -> None:
    print(f"\n--- {name} ---")
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
    print("F1-score :", f1_score(y_true, y_pred, zero_division=0))
    print("ROC-AUC  :", roc_auc_score(y_true, y_proba))
    print("Accuracy :", accuracy_score(y_true, y_pred))

In [17]:
cfg = TrainConfig()
X_train, X_test, y_train, y_test, _ = prepare_arrays()

# 1) Deep MLP 
deep_study = run_optuna_for_architecture("deep_mlp", X_train, y_train, cfg)
print("\nBest params for DeepMLP:")
print(deep_study.best_params)
print("Best validation ROC-AUC:", deep_study.best_value)

deep_pred, deep_proba = fit_final_model(
    architecture="deep_mlp",
    best_params=deep_study.best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    cfg=cfg,
)
print_metrics("DeepMLP", y_test, deep_pred, deep_proba)

cuda


[I 2026-03-30 16:25:54,935] A new study created in memory with name: deep_mlp_study
Best trial: 0. Best value: 0.879246:  10%|█         | 1/10 [01:03<09:30, 63.40s/it]

[I 2026-03-30 16:26:58,469] Trial 0 finished with value: 0.8792458032234141 and parameters: {'batch_size': 512, 'lr': 0.0001700037298921101, 'weight_decay': 2.9375384576328313e-06, 'n_layers': 2, 'hidden_dim_0': 896, 'hidden_dim_1': 640, 'dropout': 0.3832290311184182}. Best is trial 0 with value: 0.8792458032234141.


Best trial: 1. Best value: 0.882999:  20%|██        | 2/10 [02:34<10:36, 79.53s/it]

[I 2026-03-30 16:28:29,294] Trial 1 finished with value: 0.882999167907408 and parameters: {'batch_size': 512, 'lr': 0.00018559980846490597, 'weight_decay': 3.549878832196506e-06, 'n_layers': 3, 'hidden_dim_0': 640, 'hidden_dim_1': 512, 'hidden_dim_2': 384, 'dropout': 0.34474115788895177}. Best is trial 1 with value: 0.882999167907408.


...

[I 2026-03-30 16:44:04,736] Trial 7 pruned. 


Best trial: 1. Best value: 0.882999:  90%|█████████ | 9/10 [20:44<02:39, 159.47s/it]

[I 2026-03-30 16:46:39,085] Trial 8 finished with value: 0.8827015207789215 and parameters: {'batch_size': 1024, 'lr': 0.0005917520523090669, 'weight_decay': 1.9170041589170666e-05, 'n_layers': 2, 'hidden_dim_0': 128, 'hidden_dim_1': 128, 'dropout': 0.3545641645055122}. Best is trial 1 with value: 0.882999167907408.


Best trial: 9. Best value: 0.883726: 100%|██████████| 10/10 [22:54<00:00, 137.42s/it]


[I 2026-03-30 16:48:49,314] Trial 9 finished with value: 0.8837256490508847 and parameters: {'batch_size': 1024, 'lr': 0.0004038176882071838, 'weight_decay': 0.0001847793417351927, 'n_layers': 2, 'hidden_dim_0': 128, 'hidden_dim_1': 384, 'dropout': 0.16448851490160177}. Best is trial 9 with value: 0.8837256490508847.

Best params for DeepMLP:
{'batch_size': 1024, 'lr': 0.0004038176882071838, 'weight_decay': 0.0001847793417351927, 'n_layers': 2, 'hidden_dim_0': 128, 'hidden_dim_1': 384, 'dropout': 0.16448851490160177}
Best validation ROC-AUC: 0.8837256490508847

--- DeepMLP ---
Precision: 0.0999583373393584
Recall   : 0.7674704724409449
F1-score : 0.17687923554597781
ROC-AUC  : 0.8390346909377704


### Best params for DeepMLP:

- batch_size: 1024,
- lr: 0.0004038176882071838,
- weight_decay: 0.0001847793417351927,
- n_layers: 2,
- hidden_dim_0: 128,
- hidden_dim_1: 384,
- dropout: 0.16448851490160177

#### Metrics
--- DeepMLP ---

- Precision: 0.0999583373393584
- Recall   : 0.7674704724409449
- F1-score : 0.17687923554597781
- ROC-AUC  : 0.8390346909377704

In [ ]:
# 2) Residual MLP
res_study = run_optuna_for_architecture("residual_mlp", X_train, y_train, cfg)
print("\nBest params for ResidualMLP:")
print(res_study.best_params)
print("Best validation ROC-AUC:", res_study.best_value)

res_pred, res_proba = fit_final_model(
    architecture="residual_mlp",
    best_params=res_study.best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    cfg=cfg,
)
print_metrics("ResidualMLP", y_test, res_pred, res_proba)

[I 2026-03-30 16:54:53,941] A new study created in memory with name: residual_mlp_study
Best trial: 0. Best value: 0.879159:  10%|█         | 1/10 [02:41<24:11, 161.31s/it]

[I 2026-03-30 16:57:35,253] Trial 0 finished with value: 0.8791587052120993 and parameters: {'batch_size': 512, 'lr': 0.0001700037298921101, 'weight_decay': 2.9375384576328313e-06, 'width': 128, 'n_blocks': 6, 'dropout': 0.34044600469728353}. Best is trial 0 with value: 0.8791587052120993.


Best trial: 1. Best value: 0.882971:  20%|██        | 2/10 [04:47<18:43, 140.49s/it]

[I 2026-03-30 16:59:41,166] Trial 1 finished with value: 0.8829713653313412 and parameters: {'batch_size': 1024, 'lr': 0.00020589728197687926, 'weight_decay': 3.5113563139704077e-06, 'width': 256, 'n_blocks': 3, 'dropout': 0.3099025726528951}. Best is trial 1 with value: 0.8829713653313412.


Best trial: 1. Best value: 0.882971:  30%|███       | 3/10 [07:54<18:51, 161.65s/it]

...

[I 2026-03-30 17:06:35,569] Trial 3 finished with value: 0.8809942136924603 and parameters: {'batch_size': 2048, 'lr': 0.0001786013788939711, 'weight_decay': 1.5673095467235414e-06, 'width': 1024, 'n_blocks': 6, 'dropout': 0.4233589392465845}. Best is trial 1 with value: 0.8829713653313412.


Best trial: 1. Best value: 0.882971:  50%|█████     | 5/10 [15:08<16:12, 194.59s/it]

[I 2026-03-30 17:10:02,424] Trial 4 finished with value: 0.8802689025012316 and parameters: {'batch_size': 1024, 'lr': 0.00015144860262751404, 'weight_decay': 3.058656666978529e-05, 'width': 128, 'n_blocks': 6, 'dropout': 0.20351199264000677}. Best is trial 1 with value: 0.8829713653313412.


Best trial: 1. Best value: 0.882971:  60%|██████    | 6/10 [17:58<12:24, 186.09s/it]

[I 2026-03-30 17:12:52,011] Trial 5 pruned. 


In [ ]:
def find_best_threshold(y_true, y_proba):
    best_threshold = 0.5
    best_score = -1.0

    for threshold in np.arange(0.01, 1.00, 0.01):
        y_pred = (y_proba >= threshold).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_score:
            best_score = score
            best_threshold = threshold

    return best_threshold, best_score